# Agent Prompt Evals
Test and iterate on the multi-step reasoning agent's system prompt and planning prompt before touching production code.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))

import json
import re
import time

import duckdb
import httpx
import pandas as pd
from IPython.display import display

from config.settings import DB_PATH, OLLAMA_SQL_MODEL

OLLAMA_URL = "http://localhost:11434/api/chat"

con = duckdb.connect(str(DB_PATH))

print(f"Model : {OLLAMA_SQL_MODEL}")
print(f"DB    : {DB_PATH}")

Model : qwen3.6:latest
DB    : /Users/vvv/Projects/groundhog/data/db/groundhog.duckdb


## 1. Schema context
Same enriched schema the production agent uses.

In [2]:
def get_schema(con):
    tables = con.execute("SHOW TABLES").fetchdf()["name"].tolist()
    parts = []
    for table in tables:
        cols = con.execute(f"DESCRIBE {table}").fetchdf()
        col_defs = ", ".join(f"{r['column_name']} {r['column_type']}" for _, r in cols.iterrows())
        note = ""
        if table == "stock_watchlist":
            tickers = con.execute("SELECT DISTINCT ticker FROM stock_watchlist").fetchdf()["ticker"].tolist()
            note = f"  -- tickers: {', '.join(tickers)}"
        elif table == "activities":
            types = con.execute("SELECT DISTINCT activity_type FROM activities").fetchdf()["activity_type"].tolist()
            note = (
                f"  -- activity_types: {', '.join(t for t in types if t)}. "
                "This is the user's personal exercise history (runs, walks, cardio, strength). "
                "For 'recent workouts' questions, use get_recent_activities or query this table."
            )
        elif table == "workouts":
            note = (
                "  -- gym class / WOD programming descriptions (e.g. SugarWOD), NOT the user's "
                "personal exercise history. Do not use this table to answer 'my recent workouts' "
                "questions — use activities instead."
            )
        parts.append(f"{table}({col_defs}){note}")
    return "\n".join(parts)

SCHEMA = get_schema(con)
print(SCHEMA)

activities(id VARCHAR, date DATE, activity_type VARCHAR, distance_miles DECIMAL(5,2), duration_seconds INTEGER, avg_pace_seconds_per_mile INTEGER, avg_hr INTEGER, calories INTEGER, created_at TIMESTAMP, max_hr INTEGER)  -- activity_types: running, walking, strength training, cardio. This is the user's personal exercise history (runs, walks, cardio, strength). For 'recent workouts' questions, use get_recent_activities or query this table.
health_metrics(date DATE, steps INTEGER, avg_hr INTEGER, active_minutes INTEGER, created_at TIMESTAMP)
memory(id VARCHAR, fact VARCHAR, embedding FLOAT[], created_at TIMESTAMP)
reminders(id VARCHAR, title VARCHAR, state VARCHAR, due_date DATE, valid_from TIMESTAMP, valid_to TIMESTAMP, is_current BOOLEAN, created_at TIMESTAMP)
sleep_metrics(date DATE, resting_hr INTEGER, hrv INTEGER, breath_rate DECIMAL(4,1), created_at TIMESTAMP, time_to_fall_asleep_minutes INTEGER, deep_sleep_minutes INTEGER)
stock_alerts(id VARCHAR, date DATE, ticker VARCHAR, alert_t

## 2. Tool definitions
Copy of the production tool list — edit here to test changes before applying to query.py.

In [3]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "run_sql",
            "description": "Run a DuckDB SQL query and return the results.",
            "parameters": {
                "type": "object",
                "properties": {"query": {"type": "string", "description": "A valid DuckDB SQL query."}},
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_latest_price",
            "description": "Get the latest closing price for a stock ticker.",
            "parameters": {
                "type": "object",
                "properties": {"ticker": {"type": "string", "description": "Exact ticker symbol e.g. INTC, BTC-USD."}},
                "required": ["ticker"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_recent_activities",
            "description": (
                "Get the user's most recent exercise activities (runs, walks, rides, cardio, "
                "strength sessions) from the activities table. Use this for any question about "
                "'recent workouts' or exercise history — the separate 'workouts' table stores gym "
                "class/WOD programming descriptions, not personal exercise history."
            ),
            "parameters": {
                "type": "object",
                "properties": {"limit": {"type": "integer", "description": "Number of activities to return. Default 5."}},
                "required": [],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_health_summary",
            "description": "Get the health metrics (steps, avg HR, active minutes) for a specific date.",
            "parameters": {
                "type": "object",
                "properties": {"date": {"type": "string", "description": "Date in YYYY-MM-DD format."}},
                "required": ["date"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "remember",
            "description": "Save a fact or preference to persistent memory for future recall.",
            "parameters": {
                "type": "object",
                "properties": {"fact": {"type": "string", "description": "The fact or preference to remember."}},
                "required": ["fact"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "recall",
            "description": "Search persistent memory for the user's personal opinions, preferences, and stated beliefs. Do NOT use for factual data questions — use run_sql or other data tools instead.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "The topic or question to search memory for."},
                    "top_k": {"type": "integer", "description": "Number of memories to return. Default 3."},
                },
                "required": ["query"],
            },
        },
    },
]

print(f"{len(TOOLS)} tools loaded")

6 tools loaded


## 3. Edit the prompts
Tweak `SYSTEM_PROMPT` or `PLANNING_PROMPT` and re-run the cells below to see the effect.

In [4]:
SYSTEM_PROMPT = """\
You are a personal data assistant with access to tools that query a local database.
Use tools to answer the user's question. Do not guess — always call a tool to get real data.
Call only the tools needed to answer the question. Once you have enough information, stop calling tools and give your final answer.
When a tool returns data, interpret it directly and answer. Do not call additional tools to verify or recheck the result.
For comparison queries (this week vs last week, this month vs last month, etc): if only one period appears in the results, the missing period has a count of zero. State the answer directly — do not say more data is needed.
NEVER call remember() unless the user's message explicitly uses the word 'remember' or 'save'. Do not save computed results automatically.
NEVER call recall() after remember() — if you just saved a fact you already have it, there is nothing to look up.
Call recall() ONLY when the question is about the user's personal opinions, preferences, or stated beliefs — NOT for factual questions that can be answered with SQL data.
When recall() returns memories, answer ONLY based on those memories. Do not add your own knowledge, reasoning, or qualifications. State the answer directly as fact.
When the user says 'workouts', 'exercise', or 'activities' referring to things they personally did (runs, walks, cardio, strength sessions), that means the activities table / get_recent_activities tool — NOT the workouts table, which stores unrelated gym class/WOD programming text.

Database schema:
{schema}

Execution plan:
{plan}
"""

PLANNING_PROMPT = """\
You are a planning assistant. Given a question and a list of available tools,
write a concise numbered plan describing exactly which tools to call and in what order to answer the question.
Be specific: name the tool, what argument to pass, and why it's needed.
Do NOT call any tools — only write the plan.

Available tools: run_sql, get_latest_price, get_recent_activities, get_health_summary, remember, recall

Database schema:
{schema}
"""

print("Prompts defined.")

Prompts defined.


## 4. Agent runner
The same plan → execute → synthesize loop as production. Tool results are printed inline.

In [5]:
MAX_TOOL_ROUNDS = 8

def _chat(messages, tools=True):
    payload = {"model": OLLAMA_SQL_MODEL, "messages": messages, "stream": False}
    if tools:
        payload["tools"] = TOOLS
    r = httpx.post(OLLAMA_URL, json=payload, timeout=600.0)
    r.raise_for_status()
    return r.json()["message"]


def _chat_with_retry(messages, tools=True, max_retries=2):
    """Call the model, retrying if it comes back with neither a tool call nor any
    content — an empty response that would otherwise be silently accepted as the
    final answer. Seen intermittently (both qwen3:32b and qwen3.6)."""
    message = {}
    for attempt in range(max_retries + 1):
        message = _chat(messages, tools=tools)
        if message.get("tool_calls") or message.get("content", "").strip():
            return message
        if attempt < max_retries:
            print(f"  [retry] empty response from model, retrying ({attempt + 1}/{max_retries})")
    return message


def _plan(question):
    messages = [
        {"role": "system", "content": PLANNING_PROMPT.format(schema=SCHEMA)},
        {"role": "user", "content": f"Plan how to answer: {question}"},
    ]
    return _chat_with_retry(messages, tools=False).get("content", "").strip()


def _dispatch_stub(name, args):
    """Read-only stub — runs SQL, calls price/activity helpers against the live DB."""
    if name == "run_sql":
        try:
            df = con.execute(args["query"]).fetchdf()
            return df.to_string(index=False) if not df.empty else "No results."
        except Exception as e:
            return f"SQL error: {e}"
    elif name == "get_latest_price":
        ticker = args["ticker"]
        row = con.execute(
            "SELECT date, closing_price FROM stock_watchlist WHERE ticker = ? ORDER BY date DESC LIMIT 1",
            [ticker],
        ).fetchone()
        return f"{ticker} closing price on {row[0]}: ${row[1]:,.2f}" if row else f"No data for {ticker}."
    elif name == "get_recent_activities":
        df = con.execute(
            "SELECT date, activity_type, duration_seconds, avg_hr, max_hr, calories FROM activities ORDER BY date DESC LIMIT ?",
            [args.get("limit", 5)],
        ).fetchdf()
        return df.to_string(index=False) if not df.empty else "No activities."
    elif name == "get_health_summary":
        row = con.execute(
            "SELECT date, steps, avg_hr, active_minutes FROM health_metrics WHERE date = ?",
            [args["date"]],
        ).fetchone()
        return f"Date: {row[0]}, Steps: {row[1]}, Avg HR: {row[2]}, Active minutes: {row[3]}" if row else f"No data for {args['date']}."
    elif name in ("remember", "recall"):
        return f"[{name} skipped in eval notebook — read-only mode]"
    return f"Unknown tool: {name}"


def run_question(question, verbose=True):
    t0 = time.time()
    plan = _plan(question)
    tool_calls_made = []

    if verbose:
        print(f"[plan]\n{plan}\n")

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT.format(schema=SCHEMA, plan=plan),
        },
        {"role": "user", "content": question},
    ]

    for _ in range(MAX_TOOL_ROUNDS):
        message = _chat_with_retry(messages)
        tool_calls = message.get("tool_calls")

        if not tool_calls:
            content = message.get("content", "")
            match = re.search(r"\{.*\}", content, re.DOTALL)
            if match:
                try:
                    parsed = json.loads(match.group())
                    if "name" in parsed and parsed["name"] in {t["function"]["name"] for t in TOOLS}:
                        tool_calls = [{"function": {"name": parsed["name"], "arguments": parsed.get("arguments", {})}}]
                except (json.JSONDecodeError, TypeError):
                    pass

        if not tool_calls:
            answer = message.get("content", "").strip() or "The model returned an empty response after retrying — try rephrasing your question."
            elapsed = time.time() - t0
            if verbose:
                print(f"[answer]\n{answer}\n")
                print(f"tools called: {tool_calls_made}  |  {elapsed:.1f}s")
            return {"answer": answer, "tools": tool_calls_made, "elapsed": elapsed, "plan": plan}

        messages.append(message)
        for call in tool_calls:
            name = call["function"]["name"]
            args = call["function"]["arguments"]
            if isinstance(args, str):
                args = json.loads(args)
            result = _dispatch_stub(name, args)
            tool_calls_made.append(name)
            if verbose:
                print(f"[tool] {name}({args})\n  → {result[:200]}\n")
            messages.append({"role": "tool", "content": result})

    # synthesize if max rounds hit
    synthesis_messages = messages + [{
        "role": "user",
        "content": "Based only on the tool results above, give a clear and direct answer. Do not call any more tools. If a period returned no rows, it means zero."
    }]
    answer = _chat_with_retry(synthesis_messages, tools=False).get("content", "").strip() or "No response."
    elapsed = time.time() - t0
    if verbose:
        print(f"[synthesized answer]\n{answer}\n")
        print(f"tools called: {tool_calls_made}  |  {elapsed:.1f}s  (hit max rounds)")
    return {"answer": answer, "tools": tool_calls_made, "elapsed": elapsed, "plan": plan, "max_rounds": True}


print("Runner ready.")

Runner ready.


## 5. Single question — interactive
Change `QUESTION` and re-run to experiment.

In [6]:
QUESTION = "compare my workout frequency this week vs last week"

result = run_question(QUESTION)

[plan]
1. Call `get_recent_activities` with an argument specifying a duration covering at least the last two weeks (e.g., `"2 weeks"`) to retrieve exercise history for both the current and previous week.
2. Count the number of activities in the retrieved data for this week and last week separately, then compare the counts to determine the change in workout frequency.



[tool] get_recent_activities({'limit': 50})
  →       date     activity_type  duration_seconds  avg_hr  max_hr  calories
2026-05-13            cardio              2449     132     172       400
2026-05-12 strength training              2191     139



[answer]
Based on your exercise history:

**This week:** 6 workouts
- May 13: Cardio
- May 12: Strength training & Running
- May 11: Walking, Cardio & Running

**Last week:** 0 workouts

You've been much more active this week compared to last week, with a total of 6 workouts versus none.

tools called: ['get_recent_activities']  |  31.6s


## 6. Eval suite
Each test case has a `question` and a `contains` list — strings that must appear in the answer for it to pass.

In [7]:
TEST_CASES = [
    {
        "id": "workout_freq_comparison",
        "question": "compare my workout frequency this week vs last week",
        "contains": ["6", "0"],  # 6 this week, 0 last week
        "not_contains": ["more data", "unable"],
    },
    {
        "id": "recent_activities",
        "question": "show me my recent workouts",
        "contains": ["running", "cardio"],
        "not_contains": [],
    },
    {
        "id": "btc_price",
        "question": "what is the latest price of bitcoin?",
        "contains": ["$"],
        "not_contains": ["don't have", "unable"],
    },
    {
        "id": "multi_step_hr",
        "question": "what was my avg heart rate across all workouts this week?",
        "contains": ["bpm", "avg"],
        "not_contains": ["more data", "unable"],
    },
    {
        "id": "no_spurious_remember",
        "question": "how many workouts did I do this week?",
        "contains": ["6"],
        "not_contains": [],
        "tools_must_not_include": ["remember"],
    },
    {
        "id": "no_spurious_recall",
        "question": "what was my total calories burned this week?",
        "contains": [],
        "not_contains": [],
        "tools_must_not_include": ["recall"],
    },
]

print(f"{len(TEST_CASES)} test cases defined")

6 test cases defined


In [8]:
def run_eval_suite(test_cases, verbose=False):
    rows = []
    for tc in test_cases:
        print(f"Running: {tc['id']} ...", end=" ")
        result = run_question(tc["question"], verbose=verbose)
        answer = result["answer"].lower()
        tools_used = result["tools"]

        failures = []
        for s in tc.get("contains", []):
            if s.lower() not in answer:
                failures.append(f"missing '{s}'")
        for s in tc.get("not_contains", []):
            if s.lower() in answer:
                failures.append(f"contains '{s}'")
        for t in tc.get("tools_must_not_include", []):
            if t in tools_used:
                failures.append(f"called {t}()")

        passed = len(failures) == 0
        status = "PASS" if passed else "FAIL"
        print(status)

        rows.append({
            "id": tc["id"],
            "status": status,
            "tools": ", ".join(tools_used) if tools_used else "—",
            "elapsed": f"{result['elapsed']:.1f}s",
            "failures": "; ".join(failures) if failures else "",
            "answer_snippet": result["answer"][:120],
        })

    return pd.DataFrame(rows)


results_df = run_eval_suite(TEST_CASES)
print()
display(results_df[["id", "status", "tools", "elapsed", "failures"]])

Running: workout_freq_comparison ... 

FAIL
Running: recent_activities ... 

PASS
Running: btc_price ... 

PASS
Running: multi_step_hr ... 

FAIL
Running: no_spurious_remember ... 

PASS
Running: no_spurious_recall ... 

PASS



,id,status,tools,elapsed,failures
0,workout_freq_comparison,FAIL,run_sql,51.0s,missing '6'
1,recent_activities,PASS,get_recent_activities,13.2s,
2,btc_price,PASS,get_latest_price,14.6s,
3,multi_step_hr,FAIL,"run_sql, run_sql, run_sql, run_sql, run_sql, r...",31.7s,missing 'bpm'; missing 'avg'
4,no_spurious_remember,PASS,"run_sql, run_sql, get_recent_activities, run_sql",24.1s,
5,no_spurious_recall,PASS,"run_sql, run_sql",28.8s,


## 7. Full answers
Expand to see what the model actually said for each question.

In [9]:
display(results_df[["id", "status", "answer_snippet"]])

,id,status,answer_snippet
0,workout_freq_comparison,FAIL,Based on your activity history from the last 1...
1,recent_activities,PASS,"Here are your recent workouts:\n\n**May 13, 20..."
2,btc_price,PASS,"The latest price of Bitcoin (BTC-USD) was $63,..."
3,multi_step_hr,FAIL,There are no recorded workouts for this week (...
4,no_spurious_remember,PASS,You haven't recorded any workouts this week. Y...
5,no_spurious_recall,PASS,You have no recorded activities for this week ...
